# RO-058 Trajectory optimisation: every number in the Note

The robot, its limits, the transcriptions and the solvers live in `images/trajkit.py`. Every solve here starts from
a fixed first guess, so nothing is random and there is no seed to set. The whole notebook takes a few minutes.

In [1]:
import sys, math
sys.path.insert(0, 'images')
import numpy as np
import trajkit as tk
r2, r3, r4 = (lambda x: round(float(x), 2)), (lambda x: round(float(x), 3)), (lambda x: round(float(x), 4))

## 1. The drive and the stop-and-turn baseline (Overview)

In [2]:
print('start', tk.START[:2], 'facing', math.degrees(tk.START[2]), ' goal', tk.GOAL[:2], 'facing', math.degrees(tk.GOAL[2]))
print('straight line', r3(tk.LINE_LEN), 'm at', r3(math.degrees(tk.LINE_HEADING)), 'degrees')
line = np.linspace(tk.START[:2], tk.GOAL[:2], 2001)
print('straight line: gap to the table, least', r3(tk.table_dist(*line.T).min()), 'm; disk overlap', r3(-tk.clearance(*line.T).min()), 'm')
print('straight line driven from rest to rest (no turns):', r3(tk.bang_coast_bang(tk.LINE_LEN)), 's')
st = tk.stop_and_turn()
for kind, dur, *rest in st['pieces']: print(f'  {kind:5s} {dur:.3f} s')
print('stop and turn total', r3(st['T']), 's')
print('turn on the spot', tk.W_SPIN, 'rad/s; wheel limit: v + 0.1 |omega| <= 0.6')

start [2. 1.] facing 29.999999999999996  goal [3.6 3.5] facing 90.0
straight line 2.968 m at 57.381 degrees
straight line: gap to the table, least 0.185 m; disk overlap 0.065 m
straight line driven from rest to rest (no turns): 6.147 s
  turn  0.106 s
  drive 5.380 s
  turn  0.140 s
  drive 2.249 s
  turn  0.208 s
stop and turn total 8.084 s
turn on the spot 5.999999999999999 rad/s; wheel limit: v + 0.1 |omega| <= 0.6


## 2. The straight drive: shooting with one number (Figure 4)

In [3]:
for a, end, miss, grad in tk.straight_gd(steps=6):
    print(f'a = {a:.6f}  end = {end:.6f}  miss = {miss:.6f}  slope = {grad:.4f}')
print('answer: a = 3 / 25 =', 3 / 25, '; top speed 5 a =', 5 * 3 / 25)
print('two halves: end = 37.5 a1 + 12.5 a2:', 0.5 * 5 ** 2 + 5 * 5, 0.5 * 5 ** 2)

a = 0.000000  end = 0.000000  miss = -3.000000  slope = -150.0000
a = 0.090000  end = 2.250000  miss = -0.750000  slope = -37.5000
a = 0.112500  end = 2.812500  miss = -0.187500  slope = -9.3750
a = 0.118125  end = 2.953125  miss = -0.046875  slope = -2.3438
a = 0.119531  end = 2.988281  miss = -0.011719  slope = -0.5859
a = 0.119883  end = 2.997070  miss = -0.002930  slope = -0.1465
answer: a = 3 / 25 = 0.12 ; top speed 5 a = 0.6
two halves: end = 37.5 a1 + 12.5 a2: 37.5 12.5


## 3. The 8 s least-effort drive: single shooting, multiple shooting, collocation (Figures 3, 5, 6, 8)

In [4]:
single = tk.shooting(M=1)
multi = tk.shooting(M=5)
colloc = tk.collocation(T_fix=8.0)
for name, s in (('single', single), ('multiple (5 segments)', multi), ('collocation', colloc)):
    P = s['X'] if 'X' in s else s['traj']
    print(f"{name:22s} iterations {len(s['hist']) - 1:3d}  effort {s['effort']:.4f}  largest defect {np.abs(s['defects']).max():.1e} m  least gap to the table {tk.clearance(P[:, 0], P[:, 1]).min():.4f} m")
shot = tk.simulate(tk.START, tk.first_shot(30, 8.0), 8.0)
print('first shot ends at', np.round(shot[-1, :2], 3), 'missing by', r3(np.hypot(*(shot[-1, :2] - tk.GOAL[:2]))), 'm')
segs = multi['hist'][0][0]
print('multiple shooting, first gaps', [r3(np.hypot(*(segs[j][-1, :2] - segs[j + 1][0, :2]))) for j in range(4)])
X0, U0, _ = colloc['hist'][0]; F = tk.f(X0, U0); pred = X0[:-1] + 8 / 30 / 2 * (F[:-1] + F[1:])
print('collocation, first largest defect', r3(np.hypot(*(pred[:, :2] - X0[1:, :2]).T).max()), 'm')
print('simulation step', 8 / 30 / 4, 's; knot spacing', 8 / 30, 's')

single                 iterations  73  effort 0.2171  largest defect 2.7e-07 m  least gap to the table -0.0010 m
multiple (5 segments)  iterations  80  effort 0.2171  largest defect 2.6e-07 m  least gap to the table -0.0010 m
collocation            iterations  68  effort 0.2178  largest defect 2.8e-07 m  least gap to the table 0.0006 m
first shot ends at [3.036 2.617] missing by 1.048 m
multiple shooting, first gaps [0.477, 0.154, 0.077, 0.152]
collocation, first largest defect 0.121 m
simulation step 0.06666666666666667 s; knot spacing 0.26666666666666666 s


## 4. The long lever of single shooting (Section 4.3)

In [5]:
U = single['U'].copy(); eps = 1e-5
def end(U): return tk.simulate(tk.START, U, 8.0)[-1, :2]
base = end(U)
for k in (0, 29):
    Up = U.copy(); Up[k, 1] += eps
    print(f'turn rate in interval {k + 1}: end moves', r4(np.hypot(*(end(Up) - base)) / eps * 0.1), 'm per 0.1 rad/s')
Um = multi['U'].copy(); Up = Um.copy(); Up[0, 1] += eps
e0 = tk.simulate(tk.START, Um[:6], 8.0 / 5)[-1, :2]; e1 = tk.simulate(tk.START, Up[:6], 8.0 / 5)[-1, :2]
print('multiple shooting: turn rate in interval 1 moves the end of its 1.6 s segment', r4(np.hypot(*(e1 - e0)) / eps * 0.1), 'm per 0.1 rad/s')

turn rate in interval 1: end moves 0.0791 m per 0.1 rad/s
turn rate in interval 30: end moves 0.0001 m per 0.1 rad/s
multiple shooting: turn rate in interval 1 moves the end of its 1.6 s segment 0.0081 m per 0.1 rad/s


## 5. Trapezoidal collocation of the straight drive (Figures 7 and 9)

In [6]:
for N in (10, 20, 40):
    r = tk.straight_collocation(N=N)
    print(N, 'intervals: least time', r4(r['T']), 's')
r = tk.straight_collocation(N=10); h = r['T'] / 10
print('first interval: h', r4(h), 's; v', r4(r['v'][0]), '->', r4(r['v'][1]), '; a', r4(r['a'][0]), r4(r['a'][1]), '; s', r4(r['s'][1]))
print('trapezoid', r4(h / 2 * (0 + r['v'][1])), ' exact 0.5 a h^2', r4(0.5 * 0.5 * h * h))
print('exact least time', tk.bang_coast_bang())

10 intervals: least time 6.3643 s
20 intervals: least time 6.2386 s
40 intervals: least time 6.2086 s
first interval: h 0.6364 s; v 0.0 -> 0.3182 ; a 0.5 0.5 ; s 0.1013
trapezoid 0.1013  exact 0.5 a h^2 0.1013
exact least time 6.200000000000001


## 6. The least-time drive to the door pose (Figures 1, 2, 10)

In [7]:
tm = tk.collocation_time()
print('least time', r4(tm['T']), 's; iterations', tm['nit'], tm['msg'])
ts, xs = tk.drive_controls(tm['X'], tm['U'], tm['T'])
print('replayed controls: end misses by', r4(np.hypot(*(xs[-1, :2] - tk.GOAL[:2]))), 'm; heading', r3(math.degrees(xs[-1, 2] - tk.GOAL[2])), 'degrees; top speed', r4(xs[:, 3].max()))
print('least gap to the table at the knots', r4(tk.clearance(tm['X'][:, 0], tm['X'][:, 1]).min()))
print('turn rate at the first and last knot', r3(tm['U'][0, 1]), r3(tm['U'][-1, 1]), '; acceleration first and last', r3(tm['U'][0, 0]), r3(tm['U'][-1, 0]))
print('time saved against stop and turn', r2(st['T'] - tm['T']), 's; above the straight-line bound', r3(tm['T'] - tk.bang_coast_bang(tk.LINE_LEN)), 's')

least time 6.1809 s; iterations 105 Optimization terminated successfully
replayed controls: end misses by 0.0031 m; heading -0.0 degrees; top speed 0.6
least gap to the table at the knots 0.0
turn rate at the first and last knot 2.568 4.288 ; acceleration first and last 0.5 -0.5
time saved against stop and turn 1.9 s; above the straight-line bound 0.034 s


## 7. iLQR and gradient descent (Figure 11)

In [8]:
rs = tk.ilqr_straight()
print('straight drive iLQR costs', [round(c, 5) for c in rs['costs']], '; first control', r4(rs['us'][0]), '; top speed', r4(rs['xs'][:, 1].max()))
print('exact least effort for 3 m in 10 s: 12 d^2 / T^3 =', 12 * 9 / 1000, '; a(t) = 0.18 (1 - 0.2 t), a at t = h/2:', r4(0.18 * (1 - 0.2 * (10 / 60))))
il = tk.ilqr(us0=tk.first_shot(tk.ILQR_N, tk.ILQR_T))
ci = [h[2] for h in il['hist']]
print('iLQR iterations', len(ci) - 1, 'costs', [round(c, 4) for c in ci[:16]], '... final', r4(ci[-1]))
print('first within 1 percent of the final cost:', next(i for i, c in enumerate(ci) if c <= 1.01 * ci[-1]))
gd = tk.gradient_descent(us0=tk.first_shot(tk.ILQR_N, tk.ILQR_T), iters=1000)
print('gradient descent cost after 0, 13, 100, 1000 iterations:', [round(gd['costs'][i], 1) for i in (0, 13, 100, 1000)])
print('ratio after 1000:', round(gd['costs'][1000] / ci[-1]))

straight drive iLQR costs [9000000.0, 0.10812, 0.10812, 0.10812] ; first control 0.1742 ; top speed 0.4505
exact least effort for 3 m in 10 s: 12 d^2 / T^3 = 0.108 ; a(t) = 0.18 (1 - 0.2 t), a at t = h/2: 0.174


iLQR iterations 99 costs [1422936.1483, 56240.2513, 3827.4549, 1916.6085, 1897.5969, 2.6779, 2.6687, 0.8273, 0.826, 0.8141, 0.8082, 0.7968, 0.7876, 0.2327, 0.2327, 0.2324] ... final 0.2172
first within 1 percent of the final cost: 79


gradient descent cost after 0, 13, 100, 1000 iterations: [1422936.1, 174704.1, 28458.7, 2717.5]
ratio after 1000: 12513
